In [3]:
from SteerEnergyStorage.Formulations.ElectrodeFormulations import ElectrodeFormulation
from SteerEnergyStorage.Constructions.Electrodes import Cathode, Anode
from SteerEnergyStorage.Formulations.ElectrodeAssemblies import CylindricalJellyRoll
from SteerEnergyStorage.Materials.other import Terminal
from SteerEnergyStorage.Constructions.Cells import CylindricalCell
from SteerEnergyStorage.Materials.ElectrodeMaterials import CathodeMaterial, AnodeMaterial, Binder, ConductiveAdditive
from SteerEnergyStorage.Materials.CurrentCollectors import NotchedCurrentCollector
from SteerEnergyStorage.Materials.Separators import Separator
from SteerEnergyStorage.Materials.Electrolytes import Electrolyte
from SteerEnergyStorage.Constructions.Containers import CylindricalCanister, CylindricalLidAssembly, CylindricalTerminalConnector, CylindricalCase
from SteerEnergyStorage.Materials.RawMaterials import CurrentCollectorMaterial


In [6]:
cathode_current_collector_material = CurrentCollectorMaterial.from_database(name='Aluminum')
anode_current_collector_material = CurrentCollectorMaterial.from_database(name='Copper')

In [ ]:
current_collector_length = 1200

In [ ]:
# construct cathode
cathode_active_material = CathodeMaterial(name="Faradion_Gen2_4.25V", 
                                          specific_cost=11.26, 
                                          density=4, 
                                          irreversible_capacity_scaling=1, 
                                          reversible_capacity_scaling=1)

cathode_conductive_additive = ConductiveAdditive(specific_cost=9, density=1.9)

cathode_binder = Binder(specific_cost=15, density=1.7)

cathode_formulation = ElectrodeFormulation(active_materials={cathode_active_material: 89},
                                           binders={cathode_binder: 5},
                                           conductive_additives={cathode_conductive_additive: 6})

cathode_current_collector = NotchedCurrentCollector(formula="Cu", 
                                                    length=current_collector_length,
                                                    width=70,
                                                    thickness=5,
                                                    tab_width=5,
                                                    tab_length=30,
                                                    tab_spacing=10,
                                                    bare_length=0)

cathode = Cathode(formulation=cathode_formulation,
                    mass_loading=10.68,
                    current_collector=cathode_current_collector,
                    calender_density=2.60)

cathode_current_collector.get_top_down_view(width=400).show()


In [ ]:
# construct anode
anode_active_material = AnodeMaterial(name="Faradion_HC",
                                      specific_cost=14.27,
                                      density=1.50,
                                      irreversible_capacity_scaling=1,
                                      reversible_capacity_scaling=1)

anode_conductive_additive = ConductiveAdditive(specific_cost=9, 
                                               density=1.9)

anode_binder = Binder(specific_cost=10, 
                      density=1.7)

anode_formulation = ElectrodeFormulation(active_materials={anode_active_material: 88},
                                         binders={anode_binder: 3},
                                         conductive_additives={anode_conductive_additive: 9})

anode_current_collector = NotchedCurrentCollector(formula="Al", 
                                                    length=current_collector_length,
                                                    width=75,
                                                    thickness=5,
                                                    tab_width=5,
                                                    tab_length=30,
                                                    tab_spacing=30,
                                                    bare_length=30)

anode = Anode(formulation=anode_formulation,
              mass_loading=4.5,
              current_collector=anode_current_collector,
              calender_density=0.85)

figure = anode_current_collector.get_top_down_view(width=580)
#figure.update_layout(xaxis=dict(range=[1000, 1200]))
figure.show()

In [ ]:
# construct seperator
separator = Separator(thickness=16, 
                      areal_cost=0.9, 
                      density=0.4, 
                      width=75, 
                      porosity=47, 
                      fold_length=current_collector_length+100)


figure = separator.get_top_down_view(width=580)
figure.show()

In [ ]:
cylindrical_jelly_roll = CylindricalJellyRoll(anode=anode, 
                                              cathode=cathode,
                                              separator=separator, 
                                              internal_die_diameter=3)

In [ ]:
figure = cylindrical_jelly_roll.get_layup()
figure.show()

In [ ]:
cylindrical_jelly_roll.get_top_down_view(width=1100, height=800).show()

In [ ]:
cylindrical_jelly_roll.get_side_view(width=1100, height=800).show()

In [ ]:
# build the electrolyte
electrolyte = Electrolyte(specific_cost=8.94, density=1.2)

In [ ]:
# build the encapsulation
cylindrical_shell = CylindricalCanister(formula='Al',
                                        outer_diameter=21,
                                        wall_thickness=0.4,
                                        length=115)

lid_assembly = CylindricalLidAssembly(cost=0.2, mass=2, thickness=6)

cathode_terminal_collector = CylindricalTerminalCollector(formula='Al', diameter=14, thickness=1, fill_factor=0.7)
anode_terminal_collector = CylindricalTerminalCollector(formula='Al', diameter=14, thickness=1, fill_factor=0.7)

case = CylindricalCase(canister=cylindrical_shell, 
                       lid_assembly=lid_assembly,
                       cathode_terminal_collector=cathode_terminal_collector,
                       anode_terminal_collector=anode_terminal_collector)

case.get_top_down_view().show()
case.get_bottom_up_view().show()
case.get_side_view(width=500, height=600).show()


In [ ]:
cylindrical_jelly_roll.get_top_down_view(encapsulation=case ,width=1100, height=800).show()

In [ ]:
cylindrical_jelly_roll.get_side_view(encapsulation=case, width=1100, height=800).show()

In [ ]:
# build the cell
cell = CylindricalCell(electrode_assembly=cylindrical_jelly_roll,
                        electrolyte=electrolyte,
                        electrolyte_overfill=10,
                        encapsulation=case,
                        reversible_capacity=5,
                        irreversible_capacity=0.2)


In [ ]:
figure = cell.get_capacity_voltage_plot(width=900, height=600)
figure.show()

In [ ]:

print(f"The energy the cell can hold is {cell.energy} Wh")
print(f"The specific energy of the cell is {cell.specific_energy} Wh/kg")
print(f"The cost of the cell is {cell.cost} $")
print(f"The normalized cost of the cell is {cell.normalized_cost} $/kWh")
print(f"The energy density of the cell is {cell.energy_density} Wh/L")

In [ ]:
cell.cost_breakdown


In [ ]:
figure = cell.get_cost_breakdown_plot(width=600, height=600)
figure.show() 

In [ ]:
figure = cell.get_mass_breakdown_plot(width=600, height=600)
figure.show()